# Salary Prediction - v14 (Producción)
## Modelo robusto multi-algoritmo + preparado para Streamlit

**Cambios principales vs v13:**
- **Limpieza de outliers en el TARGET (`Salary`)**: se eliminan registros con salarios irreales
  (causa raíz de predicciones absurdas como unos pocos dólares al año).
- Tratamiento explícito de outliers/valores ilógicos en `Age`.
- Orden de imputación corregido (categóricas -> luego numéricas dependientes de esas categorías).
- Nuevas features: `Age_Bin`, `Experience_Bin`, `Seniority_Score`.
- **NO se crea `Salary_Per_Hour` como feature de entrada** (ver nota de fuga de datos en la Sección 3).
- **Diagnóstico explícito de `Experience`**: correlación con `Salary` + importancia de variables,
  para distinguir si el peso bajo de `Experience` es un bug o una característica real de los datos.
- **Soporte de GPU opcional** para XGBoost y LightGBM (detección automática, con fallback a CPU).
- Comparación seria entre **XGBoost, Random Forest y LightGBM** con `RandomizedSearchCV` + `KFold`.
- Split train/test estratificado por bins de salario (aproximación práctica de "Stratified K-Fold" para regresión).
- Métricas completas (MAPE, MAE, RMSE, R²) en train **y** test para detectar overfitting.
- Análisis de importancia de variables del mejor modelo (guardado también en el `.pkl` para la app).
- Guardado del modelo con metadata suficiente para reconstruir la app sin ambigüedad.

**Nota sobre el dataset:** las columnas originales son `Name, Phone_Number, Date_Of_Birth, Experience,
Role, Qualification, University, Cert, Salary`. No existe una columna de "Departamento"; `Role` representa
más bien el nivel de seniority (p. ej. Senior vs. otros). Esto se documenta también en la app de Streamlit.

In [ ]:
!pip install pandas numpy scikit-learn xgboost lightgbm matplotlib --quiet
print("Librerías instaladas")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pickle
import json as _json

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb

# LightGBM es opcional: si no se puede instalar/importar en el entorno, seguimos con 2 modelos
try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False
    print("Aviso: LightGBM no disponible, se continuará solo con XGBoost y Random Forest")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

### Configuración de GPU (opcional)

Si tienes una GPU NVIDIA disponible, XGBoost y LightGBM pueden usarla para entrenar bastante
más rápido (Random Forest de scikit-learn **no** tiene soporte de GPU nativo, así que ese
modelo siempre corre en CPU). Esta celda detecta automáticamente si hay GPU utilizable y, si
no la hay o falla, cae de vuelta a CPU sin romper el notebook.

In [ ]:
def detect_gpu():
    """Intenta usar la GPU con un fit minúsculo de XGBoost. Si falla por cualquier
    razón (no hay GPU, drivers, CUDA, build de xgboost sin soporte GPU, etc.),
    regresa False y seguimos en CPU sin detener el notebook."""
    try:
        probe = xgb.XGBRegressor(tree_method="hist", device="cuda", n_estimators=5)
        probe.fit(np.array([[0.0], [1.0], [2.0]]), np.array([0.0, 1.0, 2.0]))
        return True
    except Exception as e:
        print(f"GPU no disponible o no utilizable ({type(e).__name__}: {e}). Se usará CPU.")
        return False

USE_GPU = detect_gpu()
DEVICE = "cuda" if USE_GPU else "cpu"
print(f"Dispositivo de entrenamiento para XGBoost/LightGBM: {DEVICE.upper()}")
print("Nota: RandomForestRegressor (scikit-learn) siempre corre en CPU, sin importar esta configuración.")

## 1. Carga de Datos + Limpieza de columnas irrelevantes

In [ ]:
URL = "https://raw.githubusercontent.com/oluwole-packt/datasets/main/salary_dataset.csv"
df = pd.read_csv(URL)
print("Shape original:", df.shape)

# Columnas que no aportan valor predictivo (identificadores personales)
df = df.drop(columns=['Name', 'Phone_Number'], errors='ignore')

# Crear Age a partir de la fecha de nacimiento
df['Date_Of_Birth'] = pd.to_datetime(df['Date_Of_Birth'], format='%d/%m/%Y', errors='coerce')
df['Age'] = 2025 - df['Date_Of_Birth'].dt.year
df = df.drop(columns=['Date_Of_Birth'], errors='ignore')

print("Paso 1 completado: columnas irrelevantes eliminadas y 'Age' creada")
df.head()

## 1.1 Limpieza de outliers en el TARGET (`Salary`)

Este paso faltaba en v13/v14 inicial y es la causa más probable de predicciones absurdas
(p. ej. un sueldo anual sugerido de unos pocos dólares): si el dataset trae registros con
errores de captura en `Salary` (valores irreales, casi cero, o exageradamente altos), el
modelo **aprende de esos errores** como si fueran patrones reales. Ningún escalamiento de
`X` corrige esto, porque el problema está en `y`.

Se aplican dos criterios, documentados y basados en los datos (no se inventan números):
1. **Regla de negocio explícita**: un salario anual por debajo de `MIN_PLAUSIBLE_SALARY` no
   es un sueldo real (es un error de captura, un placeholder, o un registro corrupto).
2. **Método IQR (rango intercuartílico)**: valores fuera de `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`
   se consideran outliers estadísticos del target.

Se **eliminan las filas** (no se imputan) porque imputar el target introduce sesgo artificial;
lo correcto ante un error de captura es descartar ese registro, no inventarle un valor.

In [ ]:
print("Salary - estadísticos ANTES de limpiar:")
print(df['Salary'].describe())

MIN_PLAUSIBLE_SALARY = 1000  # ajustar según el mercado/moneda real de tu negocio

Q1 = df['Salary'].quantile(0.25)
Q3 = df['Salary'].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

# El límite inferior real es el más estricto entre la regla de negocio y el IQR,
# para no dejar pasar sueldos que son claramente errores de captura (p. ej. "$4")
effective_lower_bound = max(MIN_PLAUSIBLE_SALARY, lower_fence)

n_before = len(df)
mask_valid_salary = (df['Salary'] >= effective_lower_bound) & (df['Salary'] <= upper_fence)
removed = df.loc[~mask_valid_salary, 'Salary']

print(f"\nLímite inferior efectivo: ${effective_lower_bound:,.2f}")
print(f"Límite superior (IQR):    ${upper_fence:,.2f}")
print(f"Filas eliminadas por Salary fuera de rango: {len(removed)} de {n_before} ({len(removed)/n_before*100:.2f}%)")
if len(removed) > 0:
    print("Ejemplos de valores eliminados:", sorted(removed.tolist())[:10])

df = df.loc[mask_valid_salary].reset_index(drop=True)

print("\nSalary - estadísticos DESPUÉS de limpiar:")
print(df['Salary'].describe())

## 2. Imputación inteligente + Tratamiento de outliers en Age

Orden importante:
1. Primero imputamos las **categóricas** (moda), porque las usamos como llaves de agrupación después.
2. Después tratamos outliers de `Age` y la imputamos (mediana agrupada por `Role`).
3. Por último imputamos `Experience` (mediana agrupada por `Role` + `Qualification`).

Esto corrige un problema de v13, donde `Experience` se imputaba usando `Role`/`Qualification`
que todavía podían tener nulos en ese punto.

In [ ]:
# === 2.1 Imputación de categóricas (moda) ===
for cat_col in ['Qualification', 'Role', 'University']:
    if cat_col in df.columns:
        df[cat_col] = df[cat_col].fillna(df[cat_col].mode()[0])

df['Cert'] = df['Cert'].fillna('No')  # ausencia de dato = asumimos sin certificación

# === 2.2 Tratamiento de outliers / valores ilógicos en Age ===
MIN_VALID_AGE = 18   # edad mínima laboral razonable
MAX_VALID_AGE = 70   # edad máxima razonable para el dataset (ajustar si el negocio lo requiere)

outliers_age = ((df['Age'] < MIN_VALID_AGE) | (df['Age'] > MAX_VALID_AGE)).sum()
print(f"Registros con Age fuera de rango [{MIN_VALID_AGE}, {MAX_VALID_AGE}]: {outliers_age}")

df.loc[(df['Age'] < MIN_VALID_AGE) | (df['Age'] > MAX_VALID_AGE), 'Age'] = np.nan

# Imputación de Age: mediana por Role, con fallback a mediana global
df['Age'] = df.groupby('Role')['Age'].transform(lambda x: x.fillna(x.median()))
df['Age'] = df['Age'].fillna(df['Age'].median())

# === 2.3 Imputación de Experience (mediana agrupada) ===
df['Experience'] = df.groupby(['Role', 'Qualification'])['Experience'].transform(
    lambda x: x.fillna(x.median())
)
df['Experience'] = df['Experience'].fillna(df['Experience'].median())

# Salvaguarda adicional: Experience no puede ser mayor que Age (regla de negocio simple)
inconsistentes = (df['Experience'] > df['Age']).sum()
print(f"Registros con Experience > Age (inconsistentes): {inconsistentes}")
df.loc[df['Experience'] > df['Age'], 'Experience'] = df['Age'] - MIN_VALID_AGE
df['Experience'] = df['Experience'].clip(lower=0)

print("\nNulos restantes por columna:")
print(df.isnull().sum())

## 3. Feature Engineering

Se mantienen las variables de v13 y se agregan:
- `Age_Bin`, `Experience_Bin`: bins ordinales, útiles para capturar efectos no lineales por rango.
- `Seniority_Score`: score compuesto (formación + rol + certificación + experiencia log).

**Nota de fuga de datos (importante):** se evaluó crear `Salary_Per_Hour`, pero esa variable es una
transformación directa del *target* (`Salary`). Incluirla como feature de entrada sería fuga de datos
(data leakage): el modelo "vería" el salario disfrazado y las métricas de evaluación serían artificialmente
buenas pero inútiles en producción. Por eso, **`Salary_Per_Hour` solo se calcula en la app de Streamlit,
a partir de la predicción ya generada**, únicamente para mostrarlo al usuario (no se usa como input).

In [ ]:
# Mapeos ordinales
QUALIFICATION_MAP = {'Bsc': 1, 'Msc': 2, 'PhD': 3}
UNIVERSITY_MAP = {'Tier3': 1, 'Tier2': 2, 'Tier1': 3}

df['Qualification_Level'] = df['Qualification'].map(QUALIFICATION_MAP)
df['University_Tier'] = df['University'].map(UNIVERSITY_MAP).fillna(2)
df['Is_Senior'] = (df['Role'] == 'Senior').astype(int)
df['Has_Cert'] = (df['Cert'] == 'Yes').astype(int)

# Interacciones y ratios (v13)
df['Experience_Age_Ratio'] = df['Experience'] / (df['Age'] + 1e-6)
df['Exp_Qual_Interaction'] = df['Experience'] * df['Qualification_Level']
df['Age_Qual_Interaction'] = df['Age'] * df['Qualification_Level']
df['Experience_Log'] = np.log1p(df['Experience'])

# --- Nuevas features v14 ---
AGE_BINS = [0, 25, 35, 45, 55, 120]
EXP_BINS = [-1, 2, 5, 10, 20, 100]

df['Age_Bin'] = pd.cut(df['Age'], bins=AGE_BINS, labels=[1, 2, 3, 4, 5]).astype(int)
df['Experience_Bin'] = pd.cut(df['Experience'], bins=EXP_BINS, labels=[1, 2, 3, 4, 5]).astype(int)

df['Seniority_Score'] = (
    df['Qualification_Level'] * 2
    + df['Is_Senior'] * 3
    + df['Has_Cert'] * 1
    + df['Experience_Log']
)

# Eliminar columnas categóricas originales (ya codificadas)
df = df.drop(columns=['Qualification', 'University', 'Role', 'Cert'], errors='ignore')

print("Paso 3 completado: Feature Engineering")
print("Columnas finales:", df.columns.tolist())

### 3.1 Diagnóstico: ¿realmente `Experience` explica el salario?

Antes de entrenar nada, conviene comprobar con los datos —no con suposiciones— qué tanto se
relaciona `Experience` (y sus variables derivadas) con `Salary`. Si esta correlación ya es baja
**en los datos crudos**, es normal (no un bug) que el modelo le dé poca importancia a `Experience`
frente a otras variables como `Age` o `Qualification_Level`. Si la correlación es alta pero el
modelo entrenado le da importancia baja, ahí sí habría que sospechar de un problema en el
pipeline (por ejemplo, columnas mal alineadas).

In [ ]:
experience_related_cols = [
    'Experience', 'Experience_Log', 'Experience_Bin', 'Experience_Age_Ratio',
    'Exp_Qual_Interaction', 'Seniority_Score', 'Age', 'Qualification_Level',
]
corr_with_salary = df[experience_related_cols + ['Salary']].corr()['Salary'].drop('Salary')
corr_with_salary = corr_with_salary.sort_values(ascending=False)

print("Correlación (Pearson) de cada variable con Salary:")
print(corr_with_salary.to_string())
print("\nSi 'Experience' aparece con una correlación baja aquí, el modelo reflejará eso: no es un")
print("error de la app, es una característica real (o limitación) de este dataset.")

## 4. Preparación de datos: target, split estratificado y escalamiento

In [ ]:
# Target
y = df['Salary'].values

# X: solo columnas numéricas, sin Salary
X = df.select_dtypes(include=[np.number]).drop(columns=['Salary'], errors='ignore').copy()
feature_names = X.columns.tolist()

print("Features usadas por el modelo:")
print(feature_names)

assert "Experience" in feature_names, (
    "'Experience' no está en las columnas del modelo: revisa por qué se perdió en el pipeline."
)

# Split "estratificado" para regresión: se bina el target en cuantiles y se usa como estrato,
# esto asegura que train y test tengan una distribución de salario comparable.
salary_bins = pd.qcut(y, q=10, labels=False, duplicates='drop')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=salary_bins
)

# Escalamiento SOLO de X. Los modelos usados (XGBoost, Random Forest, LightGBM) son basados en
# árboles y no requieren escalar y; escalar el target solo agrega complejidad y riesgo de bugs
# en el inverse_transform sin ninguna ganancia de desempeño para este tipo de modelo.
scaler_X = RobustScaler()
X_train_scaled = pd.DataFrame(scaler_X.fit_transform(X_train), columns=feature_names, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler_X.transform(X_test), columns=feature_names, index=X_test.index)

print(f"\nTrain: {len(y_train)} | Test: {len(y_test)}")

## 5. Modelado: comparación de algoritmos con búsqueda de hiperparámetros

Se entrenan y optimizan **XGBoost**, **Random Forest** y **LightGBM** (si está disponible) usando
`RandomizedSearchCV` con validación cruzada `KFold(5)`. Se reportan métricas en train y test para
detectar overfitting (si train es mucho mejor que test, hay sobreajuste).

In [ ]:
def calc_metrics(y_true, y_pred):
    return {
        "MAPE": mean_absolute_percentage_error(y_true, y_pred) * 100,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

param_distributions = {
    "XGBoost": {
        "n_estimators": [400, 600, 800, 1000, 1300],
        "max_depth": [3, 4, 5, 6],
        "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "reg_alpha": [0, 0.1, 0.26, 0.5],
        "reg_lambda": [1, 1.35, 1.5, 2],
        "min_child_weight": [1, 3, 5],
    },
    "RandomForest": {
        "n_estimators": [300, 500, 700, 900],
        "max_depth": [None, 8, 12, 16, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.8],
    },
}

base_estimators = {
    "XGBoost": xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist", device=DEVICE),
    "RandomForest": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),  # sin soporte GPU en sklearn
}

if LGBM_AVAILABLE:
    param_distributions["LightGBM"] = {
        "n_estimators": [400, 600, 800, 1000],
        "num_leaves": [15, 31, 63, 90],
        "learning_rate": [0.01, 0.02, 0.03, 0.05],
        "subsample": [0.7, 0.8, 0.9],
        "colsample_bytree": [0.7, 0.8, 0.9],
        "reg_alpha": [0, 0.1, 0.3],
        "reg_lambda": [0, 0.5, 1, 1.5],
    }
    lgbm_kwargs = dict(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    if USE_GPU:
        # La GPU en LightGBM solo funciona si el paquete se compiló con soporte GPU
        # (el pip install estándar normalmente NO lo trae). Se intenta y, si falla
        # al entrenar más adelante, hay que reinstalar lightgbm con soporte GPU o
        # dejar que corra en CPU (quitando estas dos líneas).
        lgbm_kwargs["device"] = "gpu"
    base_estimators["LightGBM"] = lgb.LGBMRegressor(**lgbm_kwargs)

results = {}
fitted_models = {}

for name, estimator in base_estimators.items():
    print(f"\nOptimizando {name}...")
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_distributions[name],
        n_iter=15,
        scoring="neg_mean_absolute_error",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
    )
    try:
        search.fit(X_train_scaled, y_train)
    except Exception as e:
        if name == "LightGBM" and USE_GPU:
            print(f"Fallo entrenando LightGBM en GPU ({type(e).__name__}: {e}). "
                  f"Reintentando en CPU (probablemente el paquete no se compiló con soporte GPU)...")
            estimator.set_params(device="cpu")
            search.fit(X_train_scaled, y_train)
        else:
            raise
    best_model = search.best_estimator_
    fitted_models[name] = best_model

    y_pred_train = best_model.predict(X_train_scaled)
    y_pred_test = best_model.predict(X_test_scaled)

    results[name] = {
        "best_params": search.best_params_,
        "train": calc_metrics(y_train, y_pred_train),
        "test": calc_metrics(y_test, y_pred_test),
    }
    print(f"{name} listo. Mejores parámetros: {search.best_params_}")

## 6. Comparación de modelos

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        "Modelo": name,
        "MAE_Train": r["train"]["MAE"], "MAE_Test": r["test"]["MAE"],
        "MAPE_Train_%": r["train"]["MAPE"], "MAPE_Test_%": r["test"]["MAPE"],
        "RMSE_Train": r["train"]["RMSE"], "RMSE_Test": r["test"]["RMSE"],
        "R2_Train": r["train"]["R2"], "R2_Test": r["test"]["R2"],
    })

comparison_df = pd.DataFrame(rows).sort_values("MAE_Test").reset_index(drop=True)
print("="*100)
print("COMPARACIÓN DE MODELOS (ordenado por MAE en test)")
print("="*100)
print(comparison_df.to_string(index=False))

BEST_MODEL_NAME = comparison_df.iloc[0]["Modelo"]
best_model = fitted_models[BEST_MODEL_NAME]
print(f"\n>>> Mejor modelo: {BEST_MODEL_NAME}")

## 7. Importancia de variables (mejor modelo)

In [ ]:
import matplotlib.pyplot as plt

importances = best_model.feature_importances_
imp_df = pd.DataFrame({"Feature": feature_names, "Importance": importances}).sort_values("Importance", ascending=True)

plt.figure(figsize=(9, 7))
plt.barh(imp_df["Feature"], imp_df["Importance"], color="#2c7fb8")
plt.title(f"Importancia de variables - {BEST_MODEL_NAME}")
plt.xlabel("Importancia")
plt.tight_layout()
plt.show()

imp_df.sort_values("Importance", ascending=False)

## 8. Evaluación final del mejor modelo

In [ ]:
final_train = results[BEST_MODEL_NAME]["train"]
final_test = results[BEST_MODEL_NAME]["test"]

print("="*75)
print(f"RESULTADOS FINALES - {BEST_MODEL_NAME} (v14)")
print("="*75)
print(f"{'Métrica':<10}{'Train':>15}{'Test':>15}")
print(f"{'MAPE':<10}{final_train['MAPE']:>14.2f}%{final_test['MAPE']:>14.2f}%")
print(f"{'MAE':<10}${final_train['MAE']:>13,.0f}{'':>1}${final_test['MAE']:>13,.0f}")
print(f"{'RMSE':<10}${final_train['RMSE']:>13,.0f}{'':>1}${final_test['RMSE']:>13,.0f}")
print(f"{'R2':<10}{final_train['R2']:>15.4f}{final_test['R2']:>15.4f}")
print("="*75)
print("Si Train es mucho mejor que Test, hay señal de overfitting; en ese caso, reducir")
print("la complejidad del modelo (max_depth, n_estimators) o aumentar la regularización.")

## 9. Guardado del modelo (listo para Streamlit)

Se guarda todo lo necesario para que la app reconstruya el pipeline de features sin ambigüedad:
modelo, scaler de X, nombres/orden de columnas, mapeos categóricos, bins y métricas de error
(para mostrar el margen de error en la app).

In [ ]:
model_data = {
    "model": best_model,
    "model_name": BEST_MODEL_NAME,
    "scaler_X": scaler_X,
    "feature_names": feature_names,
    "qualification_map": QUALIFICATION_MAP,
    "university_map": UNIVERSITY_MAP,
    "age_bins": AGE_BINS,
    "exp_bins": EXP_BINS,
    "age_bounds": {"min": MIN_VALID_AGE, "max": MAX_VALID_AGE},
    "metrics": {"train": final_train, "test": final_test},
    "all_models_comparison": comparison_df.to_dict(orient="records"),
    "feature_importances": dict(zip(feature_names, [float(x) for x in importances])),
    "salary_range": {
        "min_observed": float(y_train.min()),
        "max_observed": float(y_train.max()),
        "p01": float(np.percentile(y_train, 1)),
        "p99": float(np.percentile(y_train, 99)),
    },
}

OUTPUT_PATH = "salary_model_v14.pkl"
with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(model_data, f)

print(f"Modelo guardado exitosamente como: {OUTPUT_PATH}")
print(f"Modelo ganador: {BEST_MODEL_NAME}")
print(f"MAE test: ${final_test['MAE']:,.0f} | MAPE test: {final_test['MAPE']:.2f}% | R2 test: {final_test['R2']:.4f}")

## 10. Notas finales

- **Limpieza del target (`Salary`)**: se eliminaron registros con salarios irreales (por debajo de
  `MIN_PLAUSIBLE_SALARY` o fuera del rango IQR) en la Sección 1.1. Esta es la corrección principal para
  evitar que el modelo sugiera cifras absurdas como unos pocos dólares al año: antes, esos registros
  corruptos formaban parte del entrenamiento y el modelo los trataba como patrones válidos.
- **`salary_range` en el `.pkl`**: guarda el mínimo/máximo observado y los percentiles 1 y 99 del salario
  de entrenamiento. La app usa estos valores para **advertir de forma transparente** (no para inventar
  números) cuando una combinación de datos de entrada cae fuera del rango que el modelo realmente conoce.
- **`Salary_Per_Hour`** deliberadamente no se usa como feature (fuga de datos). La app la calcula
  solo para mostrarla, dividiendo el salario **predicho** entre horas laborales anuales estándar (2080 h).
- **`Departamento`**: el dataset original no incluye esta columna. La app la solicita solo como
  dato de referencia/historial, no participa en la predicción. Si en el futuro se cuenta con un
  dataset que sí incluya departamento, se puede añadir como variable categórica (one-hot) sin
  cambiar el resto del pipeline.
- Para reentrenar con datos nuevos, basta con reemplazar la URL de carga y volver a ejecutar el
  notebook completo; el pickle final siempre queda con el mismo esquema de claves.